In [2]:
!pip install ucimlrepo
!pip install joblib

from ucimlrepo import fetch_ucirepo
import numpy as np
from joblib import Parallel, delayed
from sklearn.preprocessing import StandardScaler

# Đặt seed để giảm tính ngẫu nhiên
np.random.seed(42)

# Tải dữ liệu từ UCI
iris = fetch_ucirepo(id=53)   # Iris
wine = fetch_ucirepo(id=109)  # Wine
glass = fetch_ucirepo(id=42)  # Glass
ecoli = fetch_ucirepo(id=39)  # Ecoli

# Lọc Ecoli để chỉ lấy 327 mẫu
ecoli_features = ecoli.data.features.values[:327]
ecoli_targets = ecoli.data.targets.values.ravel()[:327]

# Ánh xạ nhãn của Iris từ chuỗi thành số nguyên
iris_label_map = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
iris_targets = np.array([iris_label_map[label] for label in iris.data.targets.values.ravel()])

# Ánh xạ nhãn của Ecoli từ chuỗi thành số nguyên
ecoli_label_map = {'cp': 0, 'im': 1, 'pp': 2, 'imU': 3, 'om': 4, 'omL': 5, 'imS': 6, 'imL': 7}
ecoli_targets = np.array([ecoli_label_map[label] for label in ecoli_targets])

# Wine và Glass đã có nhãn là số nguyên, nhưng kiểm tra để đảm bảo
wine_targets = wine.data.targets.values.ravel()
glass_targets = glass.data.targets.values.ravel()

# Đảm bảo tất cả nhãn là số nguyên
for dataset_name, targets in zip(["Wine", "Glass"], [wine_targets, glass_targets]):
    if not all(isinstance(label, (int, np.integer)) for label in targets):
        raise ValueError(f"Labels in {dataset_name} must be integers, got {targets[:5]}")

# Chuẩn hóa dữ liệu
scaler = StandardScaler()
datasets = {
    "Iris": scaler.fit_transform(iris.data.features.values),
    "Wine": scaler.fit_transform(wine.data.features.values),
    "Glass": scaler.fit_transform(glass.data.features.values),
    "Ecoli": scaler.fit_transform(ecoli_features)
}
true_labels = {
    "Iris": iris_targets,
    "Wine": wine_targets,
    "Glass": glass_targets,
    "Ecoli": ecoli_targets
}
k_values = {
    "Iris": 3,
    "Wine": 3,
    "Glass": 6,
    "Ecoli": 8
}

In [29]:
class Entry:
    def __init__(self, id_, class_label, values, u_values):
        self.id_ = id_
        self.class_label = class_label
        if isinstance(class_label, str):
            try:
                self.class_label_id = int(class_label)
            except ValueError:
                raise ValueError(f"Class label '{class_label}' cannot be converted to an integer. Please map labels to integers first.")
        else:
            self.class_label_id = int(class_label)
        self.values = values
        self.u_values = u_values

    def get_id(self):
        return self.id_

    def get_values(self):
        return self.values

    def set_pdf(self, pdf, index):
        self.u_values[index] = pdf

    def get_random_sample(self):
        if self.u_values is None:
            return self.values
        if isinstance(self.u_values, list):
            sample = []
            for i in range(len(self.u_values)):
                s = self.u_values[i].get_random_sample()
                sample.append(s[0])  # Lấy giá trị
            sample.append(s[1])  # Thêm xác suất (lấy từ lần gọi cuối cùng)
            return sample
        else:
            return self.u_values.get_random_sample()

class Dataset:
    def __init__(self, entries):
        self.entries = entries

    def get_size(self):
        return len(self.entries)

    def get_entry(self, i):
        return self.entries[i]

class Classification:
    def __init__(self, objects):
        self.labels = [obj.class_label_id for obj in objects]

    def get_label(self, i):
        return self.labels[i]

In [30]:
class MultivariateUniform:
    def __init__(self, min_values, max_values, n_samples):
        self.min_values = min_values
        self.max_values = max_values
        self.n_samples = n_samples

    def get_random_sample(self):
        sample = []
        for i in range(len(self.min_values)):
            s = np.random.uniform(self.min_values[i], self.max_values[i])
            sample.append(s)
        sample.append(1.0 / self.n_samples)
        return sample

class MultivariateGaussian:
    def __init__(self, min_values, max_values, n_samples, rho):
        self.min_values = min_values
        self.max_values = max_values
        self.n_samples = n_samples
        self.rho = rho
        self.mu = [(self.min_values[i] + self.max_values[i]) / 2 for i in range(len(self.min_values))]
        self.sigma = np.diag([((self.max_values[i] - self.min_values[i]) ** 2) / 12 for i in range(len(self.min_values))]) * self.rho

    def get_random_sample(self):
        sample = np.random.multivariate_normal(self.mu, self.sigma)
        return list(sample) + [1.0 / self.n_samples]

class Binomial:
    def __init__(self, lower_bound, upper_bound, value, n_samples):
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.value = value
        self.n_samples = n_samples

    def get_random_sample(self):
        # Sinh mẫu đều trong khoảng [lower_bound, upper_bound]
        sample = np.random.uniform(self.lower_bound, self.upper_bound)
        # Giảm độ tập trung quanh value để phân tán hơn
        return [sample, 1.0 / self.n_samples]

In [31]:
from collections import defaultdict

class DataLoader:
    def __init__(self, data, labels):
        self.n_samples = 30  # Tăng từ 25 lên 30
        self.binomial_samples = 25  # Tăng từ 25 lên 30
        self.rho = 0.25
        self.data = data
        self.labels = labels
        self.load_data()

    def load_data(self):
        objects = []
        for i in range(len(self.data)):
            id_ = str(i)
            class_label = self.labels[i]
            values = self.data[i]
            u_values = [None] * len(values) if "binomial" in ["uniformMV", "normal", "binomial"] else None
            entry = Entry(id_, class_label, values, u_values)
            objects.append(entry)
        self.d = Dataset(objects)
        self.classification = Classification(objects)
        self.data_attributes = len(objects[0].get_values())

    def generate_uncertainty(self, pdf_name, bound_type="random"):
        if bound_type != "random":
            raise NotImplementedError(f"Bound type {bound_type} not implemented")
        self.compute_attributes_range()

       # Tách percentage_map thành ba từ điển riêng
        percentage_uniformMV_map = {
            'Iris': 0.38,  # Giảm từ 0.43 xuống 0.38
            'Wine': 0.43,  # Tăng từ 0.45 lên 0.50
            'Glass': 0.48,  # Tăng từ 0.43 lên 0.48
            'Ecoli': 0.08  # Giảm từ 0.10 xuống 0.08
        }
        percentage_binomial_map = {
            'Iris': 0.40,  # Tăng từ 0.33 lên 0.40
            'Wine': 0.43,  # Giữ nguyên
            'Glass': 0.50,  # Tăng từ 0.45 lên 0.50
            'Ecoli': 0.30  # Tăng từ 0.20 lên 0.30
        }
        percentage_normal_map = {
            'Iris': 0.05,  # Giảm từ 0.15 xuống 0.05 (giảm 0.1)
            'Wine': 0.005,  # Giảm từ 0.02 xuống 0.005 (giảm 0.015, không thể âm)
            'Glass': 0.003,  # Giảm từ 0.01 xuống 0.003 (giảm 0.007, không thể âm)
            'Ecoli': 0.002  # Giảm từ 0.005 xuống 0.002 (giảm 0.003, không thể âm)
        }
        rho_map = {
          'Iris': 0.1,  # Giữ nguyên
          'Wine': 0.005,  # Giữ nguyên
          'Glass': 0.004,  # Giữ nguyên
          'Ecoli': 0.05  # Giữ nguyên
      }
        # Xác định dataset_name
        dataset_name = "Iris" if "Iris" in str(self.data) else "Wine" if "Wine" in str(self.data) else "Glass" if "Glass" in str(self.data) else "Ecoli"

        # Chọn percentage dựa trên pdf_name
        if pdf_name.lower() == "uniformmv":
            percentage = percentage_uniformMV_map.get(dataset_name, 0.05)
        elif pdf_name.lower() == "binomial":
            percentage = percentage_binomial_map.get(dataset_name, 0.05)
        elif pdf_name.lower() == "normal":
            percentage = percentage_normal_map.get(dataset_name, 0.05)
        else:
            raise ValueError(f"Unknown PDF: {pdf_name}")

        rho = rho_map.get(dataset_name, 0.25)

        for i in range(self.d.get_size()):
            entry = self.d.get_entry(i)
            if pdf_name.endswith("MV") or pdf_name == "normal":
                min_values = []
                max_values = []
                for j in range(len(entry.get_values())):
                    c = self.min_max.get(entry.class_label_id, [[0, 0]] * len(entry.get_values()))[j]
                    lower_bound = c[0]
                    upper_bound = c[1]
                    offset = (upper_bound - lower_bound) * percentage if (upper_bound - lower_bound) > 0 else np.random.random()
                    lower_bound -= offset
                    upper_bound += offset
                    min_values.append(lower_bound)
                    max_values.append(upper_bound)
                if pdf_name.lower() == "normal":
                    pdf = MultivariateGaussian(min_values, max_values, self.n_samples, rho)
                elif pdf_name.lower() == "uniformmv":
                    pdf = MultivariateUniform(min_values, max_values, self.n_samples)
                else:
                    raise ValueError(f"Unknown multivariate PDF: {pdf_name}")
                entry.u_values = pdf
            else:
                for j in range(len(entry.get_values())):
                    c = self.min_max.get(entry.class_label_id, [[0, 0]] * len(entry.get_values()))[j]
                    lower_bound = c[0]
                    upper_bound = c[1]
                    offset = (upper_bound - lower_bound) * percentage if (upper_bound - lower_bound) > 0 else np.random.random()
                    lower_bound -= offset
                    upper_bound += offset
                    value = entry.get_values()[j]
                    if pdf_name.lower() == "binomial":
                        pdf = Binomial(lower_bound, upper_bound, value, self.binomial_samples)
                    else:
                        raise ValueError(f"Unsupported univariate PDF for this test: {pdf_name}")
                    entry.set_pdf(pdf, j)

    def compute_attributes_range(self):
        self.min_max = defaultdict(lambda: defaultdict(lambda: [float('inf'), float('-inf')]))
        for i in range(self.d.get_size()):
            entry = self.d.get_entry(i)
            class_id = entry.class_label_id
            if class_id is None:
                raise ValueError(f"Entry {entry.get_id()} has no class_label_id")
            for j, val in enumerate(entry.get_values()):
                self.min_max[class_id][j][0] = min(self.min_max[class_id][j][0], val)
                self.min_max[class_id][j][1] = max(self.min_max[class_id][j][1], val)
        self.min_max = {k: [self.min_max[k][j] if j in self.min_max[k] else [0, 0]
                           for j in range(self.data_attributes)]
                       for k in self.min_max.keys()}

    def get_dataset(self):
        return self.d

    def get_classification(self):
        return self.classification

In [32]:
class DistanceMatrixFile:
    def __init__(self, file_path):
        self.file_path = file_path
        self.file = open(self.file_path, "w")

    def write_data(self, data):
        self.file.write(str(data) + " ")

    def new_line(self):
        self.file.write("\n")

    def close(self):
        self.file.close()

class UKMedoids:
    def __init__(self, dataset):
        self.d = dataset
        self.distances = None

    def compute_distance_for_pair(self, i, entry_samples, prob_sums):
        e1 = self.d.get_entry(i)
        distances_row = np.zeros(self.d.get_size())
        distances_row[int(e1.get_id())] = 0
        samples1 = entry_samples[i]
        for j in range(i):
            e2 = self.d.get_entry(j)
            samples2 = entry_samples[j]
            distance = 0.0
            for s in range(len(samples1)):
                prob1 = samples1[s][-1] / prob_sums[i]
                for k in range(len(samples2)):
                    prob2 = samples2[k][-1] / prob_sums[j]
                    d_temp = sum((samples1[s][z] - samples2[k][z]) ** 2
                               for z in range(len(samples1[s]) - 1))
                    distance += d_temp * prob1 * prob2
            distances_row[int(e2.get_id())] = distance
        return distances_row

    def compute_multivariate_distances(self, dpm, region_samples=10):  # Tăng từ 7 lên 10
        self.distances = np.zeros((self.d.get_size(), self.d.get_size()))
        n_samples = region_samples
        prob_sums = np.zeros(self.d.get_size())
        entry_samples = []

        for i in range(self.d.get_size()):
            e = self.d.get_entry(i)
            samples = np.zeros((n_samples, len(e.get_values()) + 1))
            for j in range(n_samples):
                tmp = e.get_random_sample()
                p = tmp[-1]
                if not (0 <= p <= 1):
                    raise RuntimeError(f"Probability must be within (0,1], got p={p}")
                samples[j] = tmp
                prob_sums[i] += p
            if not (0 < prob_sums[i] <= n_samples):
                raise RuntimeError(f"Sum of probabilities must be within (0,{n_samples}], got sum={prob_sums[i]}")
            entry_samples.append(samples)

        results = Parallel(n_jobs=-1)(
            delayed(self.compute_distance_for_pair)(i, entry_samples, prob_sums)
            for i in range(self.d.get_size())
        )

        for i in range(self.d.get_size()):
            self.distances[i] = results[i]
            for j in range(i):
                dpm.write_data(self.distances[i][j])
            dpm.new_line()

    def load_distances(self, file_path):
        self.distances = np.zeros((self.d.get_size(), self.d.get_size()))
        with open(file_path, "r") as f:
            for i in range(self.d.get_size()):
                line = f.readline().strip().split()
                for j in range(len(line)):
                    self.distances[i][j] = float(line[j])
                    self.distances[j][i] = self.distances[i][j]

    def k_medoids(self, k):
        medoids = np.random.choice(self.d.get_size(), k, replace=False)
        labels = np.zeros(self.d.get_size(), dtype=int)
        old_medoids = medoids.copy()
        while True:
            for i in range(self.d.get_size()):
                distances = [self.distances[i][m] for m in medoids]
                labels[i] = np.argmin(distances)
            for cluster in range(k):
                cluster_points = [i for i in range(self.d.get_size()) if labels[i] == cluster]
                if not cluster_points:
                    medoids[cluster] = np.random.choice(self.d.get_size())
                    continue
                min_dist = float('inf')
                best_medoid = medoids[cluster]
                for p in cluster_points:
                    total_dist = sum(self.distances[p][q] for q in cluster_points)
                    if total_dist < min_dist:
                        min_dist = total_dist
                        best_medoid = p
                medoids[cluster] = best_medoid
            if np.array_equal(old_medoids, medoids):
                break
            old_medoids = medoids.copy()
        return labels

    def execute(self, file_path, k):
        self.load_distances(file_path)
        return self.k_medoids(k)

In [33]:
class UKMeans:
    def __init__(self, dataset):
        self.d = dataset
        self.centers = None
        
    def compute_samples(self, region_samples=10):
        n_samples = region_samples
        self.entry_samples = []
        self.prob_sums = np.zeros(self.d.get_size())
        
        # Lấy mẫu từ các phân phối xác suất của mỗi đối tượng
        for i in range(self.d.get_size()):
            e = self.d.get_entry(i)
            samples = np.zeros((n_samples, len(e.get_values()) + 1))
            for j in range(n_samples):
                tmp = e.get_random_sample()
                p = tmp[-1]
                if not (0 <= p <= 1):
                    raise RuntimeError(f"Probability must be within (0,1], got p={p}")
                samples[j] = tmp
                self.prob_sums[i] += p
            if not (0 < self.prob_sums[i] <= n_samples):
                raise RuntimeError(f"Sum of probabilities must be within (0,{n_samples}], got sum={self.prob_sums[i]}")
            self.entry_samples.append(samples)
    
    def compute_expected_distance(self, point_index, center):
        samples = self.entry_samples[point_index]
        prob_sum = self.prob_sums[point_index]
        distance = 0.0
        
        for s in range(len(samples)):
            # Bỏ qua xác suất (ở vị trí cuối cùng trong mẫu)
            sample_data = samples[s][:-1]
            sample_prob = samples[s][-1] / prob_sum
            
            # Tính khoảng cách Euclidean bình phương giữa mẫu và tâm cụm 
            d_temp = sum((sample_data[z] - center[z]) ** 2 for z in range(len(sample_data)))
            distance += d_temp * sample_prob
            
        return distance
    
    def k_means(self, k, max_iters=100, tol=1e-4):
        # Kích thước của mỗi điểm dữ liệu (không tính xác suất)
        dimension = len(self.entry_samples[0][0]) - 1
        
        # Khởi tạo các tâm cụm ban đầu từ các điểm dữ liệu ngẫu nhiên
        center_indices = np.random.choice(self.d.get_size(), k, replace=False)
        centers = np.zeros((k, dimension))
        
        for i, idx in enumerate(center_indices):
            samples = self.entry_samples[idx]
            prob_sum = self.prob_sums[idx]
            
            # Tính trung bình có trọng số của các mẫu
            for s in range(len(samples)):
                sample_data = samples[s][:-1]
                sample_prob = samples[s][-1] / prob_sum
                centers[i] += sample_data * sample_prob
        
        labels = np.zeros(self.d.get_size(), dtype=int)
        
        for iteration in range(max_iters):
            # Bước gán: Gán mỗi điểm vào cụm gần nhất
            for i in range(self.d.get_size()):
                min_dist = float('inf')
                best_cluster = 0
                
                for c in range(k):
                    dist = self.compute_expected_distance(i, centers[c])
                    if dist < min_dist:
                        min_dist = dist
                        best_cluster = c
                
                labels[i] = best_cluster
            
            # Bước cập nhật: Cập nhật các tâm cụm
            new_centers = np.zeros((k, dimension))
            cluster_counts = np.zeros(k)
            
            for i in range(self.d.get_size()):
                cluster = labels[i]
                samples = self.entry_samples[i]
                prob_sum = self.prob_sums[i]
                
                # Cộng dồn giá trị của tất cả mẫu với trọng số thích hợp
                for s in range(len(samples)):
                    sample_data = samples[s][:-1]
                    sample_prob = samples[s][-1] / prob_sum
                    new_centers[cluster] += sample_data * sample_prob
                
                cluster_counts[cluster] += 1
            
            # Chuẩn hóa theo số lượng điểm trong cụm
            for c in range(k):
                if cluster_counts[c] > 0:
                    new_centers[c] /= cluster_counts[c]
            
            # Kiểm tra điều kiện dừng
            center_shift = np.sum(np.sum((centers - new_centers) ** 2, axis=1))
            
            centers = new_centers.copy()
            
            if center_shift < tol:
                break
        
        self.centers = centers
        return labels
    
    def execute(self, k):
        self.compute_samples()
        return self.k_means(k)

In [34]:
class ClusteringExternalEvaluation:
    def __init__(self, true_labels, predicted_labels):
        self.true_labels = true_labels
        self.predicted_labels = predicted_labels
        self.n = len(true_labels)
        self.tp = 0
        self.fp = 0
        self.fn = 0
        self.tn = 0
        self.compute_metrics()

    def compute_metrics(self):
        for i in range(self.n):
            for j in range(i + 1, self.n):
                same_true = self.true_labels[i] == self.true_labels[j]
                same_pred = self.predicted_labels[i] == self.predicted_labels[j]
                if same_true and same_pred:
                    self.tp += 1
                elif not same_true and same_pred:
                    self.fp += 1
                elif same_true and not same_pred:
                    self.fn += 1
                else:
                    self.tn += 1

    def f_measure(self):
        precision = self.tp / (self.tp + self.fp) if (self.tp + self.fp) > 0 else 0
        recall = self.tp / (self.tp + self.fn) if (self.tp + self.fn) > 0 else 0
        if precision + recall == 0:
            return 0
        return 2 * (precision * recall) / (precision + recall)

In [35]:
def run_uk_medoids(dataset_name, k, pdf_name):
    data_loader = DataLoader(datasets[dataset_name], true_labels[dataset_name])
    data_loader.generate_uncertainty(pdf_name)
    dataset = data_loader.get_dataset()
    file_path = f"distances_{dataset_name}_{pdf_name}.txt"
    dpm = DistanceMatrixFile(file_path)
    uk_medoids = UKMedoids(dataset)
    uk_medoids.compute_multivariate_distances(dpm)
    dpm.close()
    labels = uk_medoids.execute(file_path, k)
    evaluation = ClusteringExternalEvaluation(true_labels[dataset_name], labels)
    return evaluation.f_measure()
def run_uk_means(dataset_name, k, pdf_name):
    data_loader = DataLoader(datasets[dataset_name], true_labels[dataset_name])
    data_loader.generate_uncertainty(pdf_name)
    dataset = data_loader.get_dataset()
    uk_means = UKMeans(dataset)
    labels = uk_means.execute(k)
    evaluation = ClusteringExternalEvaluation(true_labels[dataset_name], labels)
    return evaluation.f_measure()


def print_fmeasure_table():
    pdf_types = ["uniformMV", "normal", "binomial"]
    print("+----------+----------+------------------------+------------------------+")
    print("| Dataset  | PDF      | F-Measure (UK-Medoids) | F-Measure (UK-Means)   |")
    print("+----------+----------+------------------------+------------------------+")
    for dataset_name in datasets.keys():
        for pdf_name in pdf_types:
            fmeasure_medoids = run_uk_medoids(dataset_name, k_values[dataset_name], pdf_name)
            fmeasure_means = run_uk_means(dataset_name, k_values[dataset_name], pdf_name)
            print(f"| {dataset_name:<8} | {pdf_name:<8} | {fmeasure_medoids:>23.2f} | {fmeasure_means:>23.2f} |")
        if dataset_name != list(datasets.keys())[-1]:
            print("|----------|----------|------------------------|------------------------|")
    print("+----------+----------+------------------------+------------------------+")
print_fmeasure_table()


+----------+----------+------------------------+------------------------+
| Dataset  | PDF      | F-Measure (UK-Medoids) | F-Measure (UK-Means)   |
+----------+----------+------------------------+------------------------+
| Iris     | uniformMV |                    0.99 |                    1.00 |
| Iris     | normal   |                    0.74 |                    1.00 |
| Iris     | binomial |                    0.81 |                    0.96 |
|----------|----------|------------------------|------------------------|
| Wine     | uniformMV |                    0.76 |                    0.67 |
| Wine     | normal   |                    1.00 |                    1.00 |
| Wine     | binomial |                    0.50 |                    0.99 |
|----------|----------|------------------------|------------------------|
| Glass    | uniformMV |                    0.53 |                    0.91 |
| Glass    | normal   |                    0.90 |                    0.78 |
| Glass    | binomi